## StateGraph实例
创建一个简单的LangGraph，包含两个节点：  
- greeter节点：问候节点，向用户打招呼
- end节点：结束节点，返回再见信息

执行流程图：
[开始] → [greeter 节点] → [end 节点] → [结束]

In [24]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

True

## State详解
在LangGraph中，State（状态）是整个图（Graph）中所有节点之间共享和传递的数据容器
**✅ 很好的问题！** 这段代码是 **LangGraph** 的核心概念之一。

| 对比          | LangChain Chain          | LangGraph                          |
|---------------|--------------------------|------------------------------------|
| 是否有记忆    | 通常无状态或手动管理     | 内置状态管理                       |
| 多节点协作    | 很难                     | 天然支持（每个节点读写同一份 State）|
| 循环、分支    | 困难                     | 非常容易（依靠 State 驱动）        |
| 可中断、可恢复| 困难                     | 原生支持（State 可以持久化）       |

**具体作用：**

- **节点之间传递数据**：一个节点执行完，把结果写进 `state["messages"]`，下一个节点就能直接拿到。
- **实现记忆功能**：因为 `messages` 一直在累积，所以 Agent 能“记住”之前的对话。
- **驱动图的运行**：LangGraph 会根据 State 的变化决定下一步去哪个节点（路由）。
- **支持持久化**：你可以把 State 保存到数据库（MemorySaver），实现“中断后继续对话”。


In [25]:
from typing import TypedDict
from langgraph.graph import StateGraph, END, START

In [26]:
# step1：定义状态结构
class State(TypedDict):
    """
    定义在节点之间传递的状态结构
    message：存储状态列表
    """
    messages: list[str]

In [27]:
# step2 定义节点函数
def greeter(state:State)->State:
    """
    问候节点：向用户打招呼
    :param state: 
    :return: State
    """
    print('--- 节点：greeter ---')
    # 在原有消息列表基础上添加新消息
    new_messages = state['messages'] + ['欢迎使用LangGraph']
    return {'messages': new_messages}

def end_node(state:State)->State:
    """
    结束节点：说再见
    :param state: 
    :return: 
    """
    print('--- 节点：end ---')
    new_messages = state['messages'] + ['感谢使用，再见']
    return {'messages': new_messages}

## INVALID_GRAPH_NODE_RETURN_VALUE
Nodes in your graph must return a dict containing one or more keys defined in your state.  
图中的节点必须返回一个字典，其中包含State中定义的一个或多个Key

example：
```python
class State(TypedDict):
    some_key: str

def bad_node(state: State):
    # Should return a dict with a value for "some_key", not a list
    return ["whoops"]

builder = StateGraph(State)
builder.add_node(bad_node)
...

graph = builder.compile()
```
调用上述图将导致如下错误：  
`graph.invoke({ "some_key": "someval" })`
```text
InvalidUpdateError: Expected dict, got ['whoops']
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE
```

参考:https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE?utm_source=chatgpt.com

In [28]:
# step3 构建图结构
def create_graph():
    """
    创建并返回编译好的 StateGraph
    """
    workflow = StateGraph(State)
    
    # 添加节点到图中
    # 第一个参数是节点名称(字符串),第二个参数是节点函数
    workflow.add_node("greeter", greeter)
    workflow.add_node("end", end_node)
    
    # 设置起始节点: 从 START 连接到 greeter
    workflow.add_edge(START, "greeter")
    
    # 添加边: 从 greeter 连接到 end
    workflow.add_edge("greeter", "end")
    
    # 添加边: 从 end 连接到 END (结束)
    workflow.add_edge("end", END)
    
    # 编译图,生成可运行的应用
    app = workflow.compile()
    
    return app

In [29]:
# step4 执行
print('--- 开始执行LangGraph ---')

# 创建图应用
app = create_graph()

# 定义初始状态
initial_state = {
    'messages': []
}

for output in app.stream(initial_state):
    # output 是一个字典,key 是节点名称,value 是该节点的输出状态
    print(output)
    


--- 开始执行LangGraph ---
--- 节点：greeter ---
{'greeter': {'messages': ['欢迎使用LangGraph']}}
--- 节点：end ---
{'end': {'messages': ['欢迎使用LangGraph', '感谢使用，再见']}}
